# 05 · Segment, fairness & escalation audit — Keelpay copilot

Companion of `05_metrics.md` (§1–§3), `06_ab_protocol.md` (§5) and `07_operations.md` (§1, §4). Every figure is computed directly from versioned repository files — no external API call:

1. **Runtime Qualifier & Flow v2 gates on the 100 hand-labelled cases** (`evals/eval_set_v1.csv` + `agentic-workflow-subproject/data/runs/regression/qualifier_eval_set_v1.csv`): contract validity, routing, silent errors, and the **missed-escalation floor** (tickets belonging to a person that would still receive a draft).
2. **Fairness and segment audit** (`05_metrics.md` §3): *selection-rate parity* (share auto-routed and share draft-eligible) and *equal error rates* (misrouting, silent errors, needless triage) across **language** (`fr` · `en` · `other`), **desk**, **urgency** and **channel/register**, compared side-by-side with the 610-case synthetic `test_natural` split.
3. **Rollback switch (`copilot_mode`, #60) replay** on the 100 cases at `l1_drafting_off`, `l2_routing_off` and `l3_copilot_off`.

In [1]:
import csv, json, math, re, tomllib
from collections import Counter
from pathlib import Path
from scipy.stats import fisher_exact

ROOT = Path.cwd()
while not (ROOT / "agentic-workflow-subproject").exists():
    ROOT = ROOT.parent
SUB = ROOT / "agentic-workflow-subproject"

with open(ROOT / "evals" / "eval_set_v1.csv", newline="", encoding="utf-8-sig") as f:
    eval_v1 = list(csv.DictReader(f))
with open(SUB / "data" / "runs" / "regression" / "qualifier_eval_set_v1.csv", newline="", encoding="utf-8") as f:
    reg_preds = {r["id"]: (r["raw_output"] or None) for r in csv.DictReader(f)}
with open(SUB / "data" / "dataset" / "test_natural.csv", newline="", encoding="utf-8") as f:
    test_nat_rows = list(csv.DictReader(f))
with open(SUB / "docs" / "results" / "d_q4_test_natural_predictions.csv", newline="", encoding="utf-8") as f:
    test_nat_preds = [r["extracted_response"] or None for r in csv.DictReader(f)]
with open(SUB / "flow" / "config.toml", "rb") as f:
    flow_cfg = tomllib.load(f)

SUB_INTENTS = {
    "payment_status": ("delayed_credit", "missing_payment", "status_inquiry", "refund_status"),
    "dispute": ("unauthorised_fee", "chargeback", "duplicate_charge", "amount_mismatch"),
    "technical_api": ("webhook_down", "api_error", "auth_failure", "integration_question"),
    "compliance": ("kyc_update", "account_hold", "document_request", "regulatory_question"),
}
DESK_OF = {"payment_status": "payments", "dispute": "disputes", "technical_api": "technical_api", "compliance": "compliance"}
SENTINELS = ("OTHER", "UNDEFINED")
KEYS = {"intent", "sub_intent", "transaction_id", "amount", "currency", "urgency", "sentiment"}
CURRENCIES = ("EUR", "USD", "GBP", "CHF", "CAD", "SEK", "DKK", "NOK", "PLN")
REF_FIELDS = ("transaction_id", "amount", "currency")
REF_CHECKS = {
    "transaction_id": lambda v: bool(re.fullmatch(r"TX-\d{6}", v)),
    "amount": lambda v: bool(re.fullmatch(r"\d+\.\d{2}", v)),
    "currency": lambda v: v in CURRENCIES,
}

def repair_references(raw):
    if not raw:
        return raw
    try:
        d = json.loads(raw)
    except (TypeError, ValueError):
        return raw
    if not isinstance(d, dict):
        return raw
    changed = False
    for k, fn in REF_CHECKS.items():
        v = d.get(k)
        if isinstance(v, str) and v not in ("", "UNDEFINED") and not fn(v):
            d[k] = "UNDEFINED"
            changed = True
    return json.dumps(d, ensure_ascii=False) if changed else raw

def parse_label(raw):
    if not raw:
        return None
    try:
        d = json.loads(raw)
    except (TypeError, ValueError):
        return None
    if not isinstance(d, dict) or set(d) != KEYS or not all(isinstance(v, str) for v in d.values()):
        return None
    i, s = d["intent"], d["sub_intent"]
    ok = (
        i in (*SUB_INTENTS, *SENTINELS)
        and (s == i if i in SENTINELS else s in (*SUB_INTENTS[i], *SENTINELS))
        and d["urgency"] in ("high", "medium", "low", "UNDEFINED")
        and d["sentiment"] in ("negative", "neutral", "positive", "UNDEFINED")
        and all(d[k] in ("", "UNDEFINED") or REF_CHECKS[k](d[k]) for k in REF_FIELDS)
    )
    return d if ok else None

def desk_of(lbl):
    return DESK_OF.get(lbl["intent"], "triage") if lbl else "triage"

def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    denom = 1 + z * z / n
    center = (p + z * z / (2 * n)) / denom
    half = (z / denom) * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (max(0.0, center - half), min(1.0, center + half))

print(f"eval_set_v1:  {len(eval_v1)} hand-labelled cases, {len(reg_preds)} stored runtime predictions")
print(f"test_natural: {len(test_nat_rows)} frozen synthetic cases, {len(test_nat_preds)} D-q4 predictions")

eval_set_v1:  100 hand-labelled cases, 100 stored runtime predictions
test_natural: 610 frozen synthetic cases, 610 D-q4 predictions


## 1 · Runtime Qualifier and Flow v2 escalation floor on `eval_set_v1` (100 cases)

On the S4 run (`evals/report_v0.md`, 50 cases), 4 of 4 tickets belonging to a person still received a model draft because Flow v2's pre-draft Re-router hold and `human_only` barrier were not wired yet (`05_metrics.md` §1–§2). Below we score the 100-case runtime predictions and pass them through the Flow v2 routing and `human_only` rules (`flow/config.toml`).

In [2]:
HUMAN_SUB = set(flow_cfg.get("human_only_sub_intents", []))
HUMAN_KW = tuple(flow_cfg.get("human_only_keywords", []))

def is_human_only_text(sub_intent, text):
    low = (text or "").lower()
    return sub_intent in HUMAN_SUB or any(kw in low for kw in HUMAN_KW)

records = []
for row in eval_v1:
    cid = row["id"]
    truth = {k: row[f"label_{k}"] for k in ("intent", "sub_intent", "transaction_id", "amount", "currency", "urgency", "sentiment")}
    raw = repair_references(reg_preds[cid])
    pred = parse_label(raw)
    t_desk, p_desk = desk_of(truth), desk_of(pred)
    routed = p_desk != "triage"
    silent = routed and (p_desk != t_desk or any(pred[f] != truth[f] for f in REF_FIELDS))
    misroute = routed and p_desk != t_desk
    needless = (not routed) and (t_desk != "triage")
    # Flow v2 draft eligibility: routed to a desk AND not blocked by human_only sub-intent/keyword
    human_blocked = routed and is_human_only_text(pred["sub_intent"], row["input_the_agent_receives"])
    gets_draft = routed and not human_blocked
    seg_head = row["segment"].split("·")[0].strip().lower()
    lang = "fr" if seg_head.startswith("fr") else ("en" if seg_head.startswith("en") else "other")
    channel = row["segment"].split("·", 1)[1].strip().lower() if "·" in row["segment"] else "email"
    std_email = channel in {"email", "formal email", "calm email", "frustrated email", "angry email", "alarmed email", "urgent email", "very polite email", "long formal email"}
    records.append({
        "id": cid, "lang": lang, "channel_group": "standard_email" if std_email else "edge_format",
        "truth": truth, "pred": pred, "valid": pred is not None,
        "t_desk": t_desk, "p_desk": p_desk, "routed": routed,
        "misroute": misroute, "silent": silent, "needless": needless,
        "human_blocked": human_blocked, "gets_draft": gets_draft,
        "belongs_triage": t_desk == "triage",
        "belongs_human_topic": t_desk == "triage" or truth["sub_intent"] in HUMAN_SUB,
    })

n = len(records)
routed_recs = [r for r in records if r["routed"]]
high_recs = [r for r in records if r["truth"]["urgency"] == "high"]
triage_truth = [r for r in records if r["belongs_triage"]]
human_truth = [r for r in records if r["belongs_human_topic"]]

print(f"Valid contract JSON:          {sum(r['valid'] for r in records):3d} / {n} = {sum(r['valid'] for r in records)/n:.1%}")
print(f"Desk accuracy (all 100):      {sum(r['p_desk']==r['t_desk'] for r in records):3d} / {n} = {sum(r['p_desk']==r['t_desk'] for r in records)/n:.1%}")
print(f"Auto-routed to a desk:        {len(routed_recs):3d} / {n} = {len(routed_recs)/n:.1%}")
print(f"Misrouted among auto-routed:  {sum(r['misroute'] for r in routed_recs):3d} / {len(routed_recs)} = {sum(r['misroute'] for r in routed_recs)/len(routed_recs):.1%}")
print(f"Silent errors (all 100):      {sum(r['silent'] for r in records):3d} / {n} = {sum(r['silent'] for r in records)/n:.1%}  ({sum(r['silent'] for r in routed_recs)}/{len(routed_recs)} = {sum(r['silent'] for r in routed_recs)/len(routed_recs):.1%} of auto-routed)")
print(f"Urgent routed to wrong desk:  {sum(r['misroute'] for r in high_recs):3d} / {len(high_recs)} = {sum(r['misroute'] for r in high_recs)/len(high_recs):.1%}")
print(f"Needless triage (over-hand):  {sum(r['needless'] for r in records):3d} / {n} = {sum(r['needless'] for r in records)/n:.1%}")
print(f"Draft-eligible after gates:   {sum(r['gets_draft'] for r in records):3d} / {n} = {sum(r['gets_draft'] for r in records)/n:.1%}  ({sum(r['human_blocked'] for r in records)} routed tickets held as HUMAN_ONLY_TOPIC)")
print(f"Triage-labelled (OTHER/UNDEF) getting a draft: {sum(r['gets_draft'] for r in triage_truth)} / {len(triage_truth)}  (routed wrongly instead of Re-router: {sum(r['routed'] for r in triage_truth)}/{len(triage_truth)})")
print(f"account_hold labelled getting a draft:         {sum(r['gets_draft'] for r in records if r['truth']['sub_intent']=='account_hold')} / {sum(1 for r in records if r['truth']['sub_intent']=='account_hold')}")
for r in triage_truth:
    if r["gets_draft"]:
        print("  missed triage escalation ->", r["id"], "pred=", r["pred"]["intent"], "/", r["pred"]["sub_intent"])

Valid contract JSON:           97 / 100 = 97.0%
Desk accuracy (all 100):       83 / 100 = 83.0%
Auto-routed to a desk:         79 / 100 = 79.0%
Misrouted among auto-routed:    9 / 79 = 11.4%
Silent errors (all 100):       12 / 100 = 12.0%  (12/79 = 15.2% of auto-routed)
Urgent routed to wrong desk:    1 / 19 = 5.3%
Needless triage (over-hand):    8 / 100 = 8.0%
Draft-eligible after gates:    69 / 100 = 69.0%  (10 routed tickets held as HUMAN_ONLY_TOPIC)
Triage-labelled (OTHER/UNDEF) getting a draft: 5 / 18  (routed wrongly instead of Re-router: 5/18)
account_hold labelled getting a draft:         0 / 7
  missed triage escalation -> NAVY-EV-16 pred= compliance / kyc_update
  missed triage escalation -> NAVY-EV-75 pred= technical_api / UNDEFINED
  missed triage escalation -> NAVY-EV-82 pred= payment_status / UNDEFINED
  missed triage escalation -> NAVY-EV-86 pred= compliance / kyc_update
  missed triage escalation -> NAVY-EV-95 pred= dispute / unauthorised_fee


## 2 · Fairness and segment breakdown (`05_metrics.md` §3)

`05_metrics.md` §3 names two fairness checks before any pilot:
- **Selection-rate parity**: do French and English tickets get auto-routed — and receive a draft after the `human_only` gate — at comparable rates?
- **Equal error rates across segments**: are misrouting, silent errors, and Re-router over-abstention (`needless_triage`) comparable between French and English, and across desks and formats?

In [3]:
def summarize_group(rows):
    m = len(rows)
    if m == 0:
        return {}
    r_cnt = sum(r["routed"] for r in rows)
    d_cnt = sum(r["gets_draft"] for r in rows)
    acc = sum(r["p_desk"] == r["t_desk"] for r in rows)
    mis = sum(r["misroute"] for r in rows)
    sil = sum(r["silent"] for r in rows)
    ndl = sum(r["needless"] for r in rows)
    urg_tot = sum(r["truth"]["urgency"] == "high" for r in rows)
    urg_bad = sum(r["misroute"] and r["truth"]["urgency"] == "high" for r in rows)
    return {
        "n": m, "routed": r_cnt, "routed_rate": r_cnt / m,
        "draft": d_cnt, "draft_rate": d_cnt / m,
        "desk_acc": acc / m,
        "misroute_of_routed": mis / r_cnt if r_cnt else 0.0, "mis": mis,
        "silent_rate": sil / m, "sil": sil,
        "needless_rate": ndl / m, "ndl": ndl,
        "urg": f"{urg_bad}/{urg_tot}",
    }

print("=== A. Language segments on eval_set_v1 (100 hand-written cases) ===")
print(f"{'lang':<8} {'n':>3}  {'auto-routed (95% CI)':<24} {'draft-elig.':<13} {'desk acc':>9} {'misroute/rtd':>13} {'silent err':>11} {'needless':>9} {'urg wrong':>9}")
by_lang = {}
for lg in ("fr", "en", "other"):
    sub = [r for r in records if r["lang"] == lg]
    s = summarize_group(sub)
    by_lang[lg] = s
    lo, hi = wilson_ci(s["routed"], s["n"])
    print(f"{lg:<8} {s['n']:>3}  {s['routed_rate']:>5.1%} [{lo:>4.0%}, {hi:>4.0%}]     {s['draft_rate']:>5.1%} ({s['draft']:>2}/{s['n']:<2}) {s['desk_acc']:>8.1%} {s['mis']:>3}/{s['routed']:<2} ({s['misroute_of_routed']:>4.1%}) {s['sil']:>2}/{s['n']:<2} ({s['silent_rate']:>4.1%}) {s['needless_rate']:>8.1%} {s['urg']:>9}")

fr_s, en_s = by_lang["fr"], by_lang["en"]
_, p_route = fisher_exact([[fr_s["routed"], fr_s["n"] - fr_s["routed"]], [en_s["routed"], en_s["n"] - en_s["routed"]]])
_, p_draft = fisher_exact([[fr_s["draft"], fr_s["n"] - fr_s["draft"]], [en_s["draft"], en_s["n"] - en_s["draft"]]])
_, p_silent = fisher_exact([[fr_s["sil"], fr_s["n"] - fr_s["sil"]], [en_s["sil"], en_s["n"] - en_s["sil"]]])
print(f"\nFR vs EN Fisher exact p-values: auto-routed p = {p_route:.3f}, draft-eligible p = {p_draft:.3f}, silent error p = {p_silent:.3f}")

print("\n=== B. Target desk segments on eval_set_v1 ===")
print(f"{'true_desk':<15} {'n':>3}  {'auto-routed':>11}  {'desk acc':>9}  {'silent err':>12}  {'needless':>9}  {'urg wrong':>9}")
for d in ("payments", "disputes", "technical_api", "compliance", "triage"):
    sub = [r for r in records if r["t_desk"] == d]
    s = summarize_group(sub)
    print(f"{d:<15} {s['n']:>3}  {s['routed']:>3}/{s['n']:<2} ({s['routed_rate']:>4.0%})  {s['desk_acc']:>8.1%}  {s['sil']:>2}/{s['n']:<2} ({s['silent_rate']:>4.1%})  {s['needless_rate']:>8.1%}  {s['urg']:>9}")

print("\n=== C. Channel / format difficulty on eval_set_v1 ===")
for grp in ("standard_email", "edge_format"):
    sub = [r for r in records if r["channel_group"] == grp]
    s = summarize_group(sub)
    print(f"{grp:<15} n={s['n']:>2}  auto-routed={s['routed_rate']:>5.1%}  desk_acc={s['desk_acc']:>5.1%}  silent_err={s['silent_rate']:>5.1%}  needless={s['needless_rate']:>5.1%}")

=== A. Language segments on eval_set_v1 (100 hand-written cases) ===
lang       n  auto-routed (95% CI)     draft-elig.    desk acc  misroute/rtd  silent err  needless urg wrong
fr        40  82.5% [ 68%,  91%]     72.5% (29/40)    85.0%   4/33 (12.1%)  6/40 (15.0%)     5.0%       1/9
en        56  75.0% [ 62%,  84%]     66.1% (37/56)    80.4%   5/42 (11.9%)  6/56 (10.7%)    10.7%       0/8
other      4  100.0% [ 51%, 100%]     75.0% ( 3/4 )   100.0%   0/4  (0.0%)  0/4  (0.0%)     0.0%       0/2

FR vs EN Fisher exact p-values: auto-routed p = 0.458, draft-eligible p = 0.656, silent error p = 0.548

=== B. Target desk segments on eval_set_v1 ===
true_desk         n  auto-routed   desk acc    silent err   needless  urg wrong
payments         25   23/25 ( 92%)     92.0%   0/25 (0.0%)      8.0%        0/1
disputes         25   22/25 ( 88%)     80.0%   4/25 (16.0%)     12.0%        1/6
technical_api    15   13/15 ( 87%)     80.0%   1/15 (6.7%)     13.3%        0/5
compliance       17   16/

In [4]:
# Compare against the 610 frozen synthetic cases of test_natural (D-q4 historical benchmark)
nat_records = []
for row, raw in zip(test_nat_rows, test_nat_preds):
    truth = json.loads(row["target"])
    pred = parse_label(raw)
    t_desk, p_desk = desk_of(truth), desk_of(pred)
    routed = p_desk != "triage"
    silent = routed and (p_desk != t_desk or any(pred[f] != truth[f] for f in REF_FIELDS))
    misroute = routed and p_desk != t_desk
    needless = (not routed) and (t_desk != "triage")
    lg = row.get("language", "").lower()
    lang = lg if lg in ("fr", "en") else "other"
    nat_records.append({
        "lang": lang, "truth": truth, "pred": pred, "t_desk": t_desk, "p_desk": p_desk,
        "routed": routed, "gets_draft": routed and not is_human_only_text(pred["sub_intent"], row["ticket"]),
        "misroute": misroute, "silent": silent, "needless": needless,
    })

print("=== Language parity on test_natural (610 synthetic cases, D-q4) ===")
print(f"{'lang':<8} {'n':>4}  {'auto-routed':>12}  {'draft-elig.':>12}  {'desk acc':>9}  {'silent err':>12}  {'urg wrong':>9}")
for lg in ("fr", "en", "other"):
    sub = [r for r in nat_records if r["lang"] == lg]
    s = summarize_group(sub)
    print(f"{lg:<8} {s['n']:>4}  {s['routed_rate']:>11.1%}  {s['draft_rate']:>11.1%}  {s['desk_acc']:>8.1%}  {s['sil']:>2}/{s['n']:<3} ({s['silent_rate']:>4.1%})  {s['urg']:>9}")

=== Language parity on test_natural (610 synthetic cases, D-q4) ===
lang        n   auto-routed   draft-elig.   desk acc    silent err  urg wrong
fr        309        82.2%        71.2%     97.1%   4/309 (1.3%)       1/88
en        291        83.2%        70.1%     98.3%   0/291 (0.0%)       0/78
other      10        70.0%        70.0%     90.0%   0/10  (0.0%)        0/1


## 3 · Rollback switch (`copilot_mode`, #60) replay on `eval_set_v1`

Before any demo or release, `04_architecture.md` §6 and `07_operations.md` §4 require running `eval_set_v1.csv` with `copilot_mode = "l1_drafting_off"` (zero model drafts) and `"l3_copilot_off"` (zero model/tool calls, manual FIFO queue operational). This cell invokes `flow.rollback_check` directly when run inside the `agentic-workflow-subproject` uv environment, and otherwise verifies the exact status invariants from the parsed predictions above.

In [5]:
import sys
sys.path.insert(0, str(SUB))
from flow.rollback_check import render_markdown, replay_mode

summaries = [replay_mode(m) for m in ("l1_drafting_off", "l2_routing_off", "l3_copilot_off")]
print(render_markdown(summaries))

# Rollback verification report (`copilot_mode`, #60)

| Mode | Tickets | Qualifier calls | Tool calls | Drafter calls | Model drafts | Auto-sends | Status split | FIFO | Human resolution |
|---|---:|---:|---:|---:|---:|---:|---|---|---|
| `l1_drafting_off` | 100 | 100 | 69 | 0 | 0 | 0 | `awaiting_reroute`=21, `awaiting_validation`=79 | yes | verified |
| `l2_routing_off` | 100 | 100 | 0 | 0 | 0 | 0 | `awaiting_reroute`=100 | yes | verified |
| `l3_copilot_off` | 100 | 0 | 0 | 0 | 0 | 0 | `awaiting_reroute`=100 | yes | verified |
